In [1]:
import polars as pl

In [5]:
df = (
    pl
    .read_parquet("/mnt/Fast2T/data/heh/0.parquet")
    .select("text")
    .filter(~pl.col("text").str.starts_with("client challenge javascript disabled you"))
)

In [6]:
df.write_csv("raw.csv")

In [10]:
(
    df
    .with_columns(words=pl.col("text").str.count_matches(r" "))
    .filter(pl.col("words") > 200)
    .with_row_index("id")
    .select(
        "id",
        pl.col("text").str.split(" ").alias("term"),
    )
    .with_columns(
        pl.col("term").list.eval(
            pl.element().filter(
                ~pl.element().str.contains(
                    r'[.,+*#!"§$%&/()=?<>^°\d\n\[\]{}]'
                )
            )
        )
    )
    .with_columns(
        pl.col("term").list.len().alias("total")
    )
    .explode("term")
    .group_by("id", "term", "total")
    .len()
    .rename({"len": "count"})
    .select("id", "term", "count", "total")
).write_csv("overcooked.json")